# RL DRC Corrector — Training

## How it works

```
User circuit (fixed topology)
        │
        ▼
  ┌──────────────────────────────────────────┐
  │  DRCCorrectorEnv (gymnasium.Env)         │
  │                                          │
  │  observation (11-dim):                   │
  │    [0-4]  DRC errors per category        │
  │           (NW, CO, PL, DF, other) / 300  │
  │    [5]    with_tie        (0 / 1)        │
  │    [6]    with_dummy      (0 / 1)        │
  │    [7]    placement       (col=1, row=0) │
  │    [8]    sep_mult  / 4.0               │
  │    [9]    met_layer / 5.0   ← routing   │
  │    [10]   width_mult / 3.0  ← routing   │
  │                                          │
  │  action (6-dim, continuous [-1, 1]):     │
  │    [0]  flip with_tie   if |a| > 0.5    │
  │    [1]  flip with_dummy if |a| > 0.5    │
  │    [2]  flip placement  if |a| > 0.5    │
  │    [3]  nudge sep_mult   ± 0.5          │
  │    [4]  nudge met_layer  ± 1            │
  │    [5]  nudge width_mult ± 0.25         │
  │                                          │
  │  reward = −DRC_errors / 300             │
  │           + 20 bonus when clean          │
  └──────────────────────────────────────────┘
        │  trains on all 15 circuits
        ▼
   PPO model  (stable-baselines3)
   saved as drc_corrector_ppo.pt
        │
        ▼
  gl.auto_build() → DRC-clean layout
```

**The circuit topology is NEVER changed.**
The RL adjusts both device-level placement and routing-level params.

- **Device params**: `with_tie`, `with_dummy`, `placement`, `sep_mult`
- **Routing params**: `met_layer` (1–3), `width_mult` (0.5–3.0)

## Prerequisites
```
pip install gymnasium>=0.29.0 stable-baselines3>=2.3.0 torch
```


In [1]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

print('All imports OK')
print('gl.auto_build available:', hasattr(gl, 'auto_build'))


All imports OK
gl.auto_build available: True


## Step 1 — Builder functions

Each circuit needs a **builder function** with this signature:

```python
def build_<circuit>(circuit_params, layout_params, name) -> Layout
```

- `circuit_params` — transistor sizing (w, fingers…) — **fixed** during training
- `layout_params`  — geometry choices the RL controls
- Returns a `Layout` that can call `.drc()`

The RL will call your builder function hundreds of times,
each time with a different `layout_params`, until DRC is clean.


In [2]:
# ── Helper: unpack layout params ────────────────────────────────────────────

def lp(p):
    """Unpack device params dict into keyword args."""
    return dict(
        with_tie   = p.get('with_tie',   False),
        with_dummy = p.get('with_dummy', False),
    )

def bp(p):
    """Unpack build() keyword args (placement, spacing, routing)."""
    return dict(
        placement  = p.get('placement',  'column'),
        sep_mult   = p.get('sep_mult',   2.0),
        met_layer  = p.get('met_layer',  1),
        width_mult = p.get('width_mult', 1.0),
    )


# ── 01 Inverter ──────────────────────────────────────────────────────────────

def build_inverter(cp, p, name='inverter'):
    vin  = gl.Net('vin');  vout = gl.Net('vout')
    mn = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vin, d=vout, s=gl.gnd, **lp(p))
    mp = gl.pmos(w=cp['wp'], fingers=cp['fn'], g=vin, d=vout, s=gl.vdd, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))


# ── 02 Current Mirror ────────────────────────────────────────────────────────

def build_current_mirror(cp, p, name='cmirror'):
    vbias = gl.Net('vbias');  iout = gl.Net('iout')
    n_or_p = cp.get('n_or_p', 'n')
    fet  = gl.nmos if n_or_p == 'n' else gl.pmos
    rail = gl.gnd  if n_or_p == 'n' else gl.vdd
    m_ref  = fet(w=cp['w'], fingers=1,                 g=vbias, d=vbias, s=rail, **lp(p))
    m_copy = fet(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=iout,  s=rail, **lp(p))
    return gl.build(m_ref, m_copy, name=name, **bp(p))


# ── 03 Differential Pair ──────────────────────────────────────────────────────

def build_diff_pair(cp, p, name='diff_pair'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail')
    vop = gl.Net('vop');  vom = gl.Net('vom')
    vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2,        g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vim,   d=vom,   s=vtail,  **lp(p))
    return gl.build(mt, mn_p, mn_m, name=name, **bp(p))


# ── 04 OTA (5T Telescopic) ───────────────────────────────────────────────────

def build_ota(cp, p, name='ota'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail');  vout = gl.Net('vout');  vleft = gl.Net('vleft')
    vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2,        g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'], fingers=cp['fn'],  g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'], fingers=cp['fn'],  g=vleft, d=vout,  s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))


# ── 05 Flipped Voltage Follower ───────────────────────────────────────────────

def build_fvf(cp, p, name='fvf'):
    vin  = gl.Net('vin');  vout = gl.Net('vout');  vfb = gl.Net('vfb')
    mn  = gl.nmos(w=cp['w_main'], fingers=2, g=vfb,  d=vout, s=gl.gnd, **lp(p))
    mp  = gl.pmos(w=cp['w_fb'],  fingers=1, g=vin,  d=vfb,  s=gl.vdd, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))


# ── 06 Transmission Gate ──────────────────────────────────────────────────────

def build_tgate(cp, p, name='tgate'):
    vin = gl.Net('vin');  vout = gl.Net('vout')
    vctrl = gl.Net('vctrl');  vctrl_n = gl.Net('vctrl_n')
    mn = gl.nmos(w=cp['wn'], fingers=1, g=vctrl,   d=vout, s=vin, **lp(p))
    mp = gl.pmos(w=cp['wp'], fingers=1, g=vctrl_n, d=vout, s=vin, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))


# ── 07 Stacked Current Mirror ─────────────────────────────────────────────────

def build_stacked_cmirror(cp, p, name='stacked_cm'):
    vbias = gl.Net('vbias');  iout = gl.Net('iout');  vcasc = gl.Net('vcasc')
    m_ref = gl.nmos(w=cp['w'], fingers=1,                  g=vbias, d=vcasc, s=gl.gnd, **lp(p))
    m_csc = gl.nmos(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=vbias, s=vcasc,  **lp(p))
    m_out = gl.nmos(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=iout,  s=gl.gnd, **lp(p))
    return gl.build(m_ref, m_csc, m_out, name=name, **bp(p))


# ── 08 Low-Voltage Current Mirror ─────────────────────────────────────────────

def build_lvcmirror(cp, p, name='lvcm'):
    vbias = gl.Net('vbias');  iout = gl.Net('iout');  vx = gl.Net('vx')
    m_ref  = gl.nmos(w=cp['w'],        fingers=1, g=vbias, d=vbias, s=gl.gnd, **lp(p))
    m_aux  = gl.nmos(w=cp['w_narrow'], fingers=1, g=vbias, d=vx,    s=gl.gnd, **lp(p))
    m_copy = gl.nmos(w=cp['w'],        fingers=1, g=vbias, d=iout,  s=gl.gnd, **lp(p))
    return gl.build(m_ref, m_aux, m_copy, name=name, **bp(p))


# ── 09 Diff Pair + CM Bias ────────────────────────────────────────────────────

def build_diff_pair_cmbias(cp, p, name='dp_cmbias'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail');  vop = gl.Net('vop');  vom = gl.Net('vom')
    vbias = gl.Net('vbias');  vload = gl.Net('vload')
    mt   = gl.nmos(w=cp['wt'], fingers=2,        g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vim,   d=vom,   s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wn'], fingers=cp['fn'],  g=vload, d=vload, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wn'], fingers=cp['fn'],  g=vload, d=vop,   s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))


# ── 10 Diff Pair + Stacked Bias ───────────────────────────────────────────────

def build_diff_pair_stacked(cp, p, name='dp_stacked'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail');  vop = gl.Net('vop');  vom = gl.Net('vom')
    vbias = gl.Net('vbias');  vcasc = gl.Net('vcasc')
    mt   = gl.nmos(w=cp['w'],  fingers=2,        g=vbias, d=vcasc, s=gl.gnd, **lp(p))
    mc   = gl.nmos(w=cp['w'],  fingers=2,        g=vbias, d=vtail, s=vcasc,  **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'],  g=vim,   d=vom,   s=vtail,  **lp(p))
    return gl.build(mt, mc, mn_p, mn_m, name=name, **bp(p))


# ── 11 Two-Stage OTA ─────────────────────────────────────────────────────────

def build_twostage_ota(cp, p, name='twostage_ota'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail');  vout = gl.Net('vout');  vleft = gl.Net('vleft')
    vbias = gl.Net('vbias');  vcs = gl.Net('vcs')
    mt   = gl.nmos(w=cp['wt'],  fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'],  fingers=2, g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'],  fingers=2, g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'],  fingers=2, g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'],  fingers=2, g=vleft, d=vout,  s=gl.vdd, **lp(p))
    mcs  = gl.nmos(w=cp['wcs'], fingers=2, g=vout,  d=vcs,   s=gl.gnd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, mcs, name=name, **bp(p))


# ── 12 P-Block (PMOS-only bias) ───────────────────────────────────────────────

def build_p_block(cp, p, name='p_block'):
    vbias = gl.Net('vbias');  iout = gl.Net('iout');  vtail = gl.Net('vtail')
    mp_ref = gl.pmos(w=cp['wp'], fingers=2, g=vbias, d=vbias, s=gl.vdd, **lp(p))
    mp_out = gl.pmos(w=cp['wp'], fingers=2, g=vbias, d=iout,  s=gl.vdd, **lp(p))
    mt     = gl.nmos(w=cp['wt'], fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_out = gl.nmos(w=cp['wn'], fingers=2, g=vbias, d=iout,  s=vtail,  **lp(p))
    return gl.build(mp_ref, mp_out, mt, mn_out, name=name, **bp(p))


# ── 13 Differential-to-Single Ended ──────────────────────────────────────────

def build_diff_to_single(cp, p, name='d2s'):
    vip = gl.Net('vip');  vim = gl.Net('vim')
    vtail = gl.Net('vtail');  vout = gl.Net('vout');  vleft = gl.Net('vleft')
    vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=2, g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=2, g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'], fingers=2, g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'], fingers=2, g=vleft, d=vout,  s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))


print('All 13 builder functions defined.')


All 13 builder functions defined.


## Step 2 — Circuit pool

All 15 circuits: `(builder_fn, circuit_params, label)`

The RL will randomly sample one circuit each episode.
This forces it to learn **general DRC rules** that work for all circuit types.


In [3]:
import random

CIRCUIT_POOL = [
    # builder_fn,              circuit_params,                                     label
    (build_inverter,           {'wn': 2.0, 'wp': 4.0, 'fn': 2},                  'inverter'),
    (build_inverter,           {'wn': 4.0, 'wp': 8.0, 'fn': 4},                  'inverter_large'),
    (build_current_mirror,     {'w': 4.0, 'ratio': 1, 'n_or_p': 'n'},            'cmirror_n'),
    (build_current_mirror,     {'w': 4.0, 'ratio': 2, 'n_or_p': 'p'},            'cmirror_p'),
    (build_diff_pair,          {'wn': 3.0, 'wt': 4.0, 'fn': 2},                  'diff_pair'),
    (build_fvf,                {'w_main': 6.6, 'w_fb': 3.3},                      'fvf'),
    (build_tgate,              {'wn': 2.0, 'wp': 2.0},                            'tgate'),
    (build_ota,                {'wn': 3.0, 'wp': 4.0, 'wt': 4.0, 'fn': 2},       'ota'),
    (build_stacked_cmirror,    {'w': 4.0, 'ratio': 1},                            'stacked_cm'),
    (build_lvcmirror,          {'w': 4.0, 'w_narrow': 1.5},                       'lvcm'),
    (build_diff_pair_cmbias,   {'wn': 3.0, 'wt': 4.0, 'fn': 2},                  'dp_cmbias'),
    (build_diff_pair_stacked,  {'wn': 3.0, 'w': 4.0, 'fn': 2},                   'dp_stacked'),
    (build_twostage_ota,       {'wn': 3.0, 'wp': 4.0, 'wt': 4.0, 'wcs': 8.0},   'twostage_ota'),
    (build_p_block,            {'wp': 3.0, 'wn': 4.0, 'wt': 4.0},                'p_block'),
    (build_diff_to_single,     {'wn': 3.0, 'wp': 4.0, 'wt': 4.0},                'd2s'),
]

print(f'Circuit pool: {len(CIRCUIT_POOL)} circuits')
for fn, cp, label in CIRCUIT_POOL:
    print(f'  {label}')


Circuit pool: 15 circuits
  inverter
  inverter_large
  cmirror_n
  cmirror_p
  diff_pair
  fvf
  tgate
  ota
  stacked_cm
  lvcm
  dp_cmbias
  dp_stacked
  twostage_ota
  p_block
  d2s


## Step 3 — DRC Corrector Environment

The environment wraps the circuit pool.
Each **episode** = one randomly picked circuit.
Each **step** = rebuild with new layout params + run Magic DRC.

### Observation (11-dim, all values in [0, 1])
| Dim  | Meaning |
|------|---------|
| 0    | NW rule errors / 300 |
| 1    | CO rule errors / 300 |
| 2    | PL rule errors / 300 |
| 3    | DF rule errors / 300 |
| 4    | other errors / 300   |
| 5    | `with_tie`  (0 or 1) |
| 6    | `with_dummy` (0 or 1) |
| 7    | `placement`  (column=1, row=0) |
| 8    | `sep_mult` / 4.0 |
| 9    | `met_layer` / 5.0  ← routing layer |
| 10   | `width_mult` / 3.0 ← wire width multiplier |

### Action (6-dim continuous, clipped to [−1, 1])
| Dim | Effect | Range |
|-----|--------|-------|
| 0   | flip `with_tie` if \|a\| > 0.5 | boolean toggle |
| 1   | flip `with_dummy` if \|a\| > 0.5 | boolean toggle |
| 2   | flip `placement` if \|a\| > 0.5 | row ↔ column |
| 3   | nudge `sep_mult` by a × 0.5 | clamp [1.0, 4.0] |
| 4   | nudge `met_layer` by round(a) | clamp [1, 3] |
| 5   | nudge `width_mult` by a × 0.25 | clamp [0.5, 3.0] |

### Reward
```
reward = −total_errors / 300   (always negative, push toward 0)
       + 20.0                   (bonus only when 0 DRC errors)
```


In [4]:
class DRCCorrectorEnv(gym.Env):
    """Multi-circuit DRC correction environment.

    Observation (11-dim):
      [0..4]  DRC errors per category (NW/CO/PL/DF/other) / MAX_ERRORS
      [5]     with_tie
      [6]     with_dummy
      [7]     placement (0=row, 1=column)
      [8]     sep_mult / 4.0
      [9]     met_layer / 5.0   ← routing layer
      [10]    width_mult / 3.0  ← routing wire width

    Action (6-dim, continuous [-1, 1]):
      [0]  |a|>0.5 → flip with_tie
      [1]  |a|>0.5 → flip with_dummy
      [2]  |a|>0.5 → flip placement
      [3]  nudge sep_mult   (±0.5, clamped 1.0–4.0)
      [4]  nudge met_layer  (±1,   clamped 1–3)
      [5]  nudge width_mult (±0.25, clamped 0.5–3.0)
    """

    RULE_CATS = ['NW', 'CO', 'PL', 'DF', 'other']
    MAX_ERRORS = 300.0

    DEFAULT_LP = {
        'with_tie':   False,
        'with_dummy': False,
        'placement':  'column',
        'sep_mult':   2.0,
        'met_layer':  1,
        'width_mult': 1.0,
    }

    def __init__(self, circuit_pool, max_steps=15):
        super().__init__()
        self.circuit_pool = circuit_pool
        self.max_steps = max_steps

        # 5 DRC categories + 6 layout param values
        self.observation_space = spaces.Box(0.0, 1.0, shape=(11,), dtype=np.float32)
        # 6 layout param deltas (device + routing)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(6,), dtype=np.float32)

        self._lp = dict(self.DEFAULT_LP)
        self._step = 0
        self._best_errors = 9999
        self.best_layout_params = dict(self.DEFAULT_LP)
        self._current_label = ''

    # ── layout param helpers ──────────────────────────────────────────────────

    def _lp_obs(self):
        return np.array([
            float(self._lp['with_tie']),
            float(self._lp['with_dummy']),
            1.0 if self._lp['placement'] == 'column' else 0.0,
            self._lp['sep_mult']   / 4.0,
            self._lp['met_layer']  / 5.0,
            self._lp['width_mult'] / 3.0,
        ], dtype=np.float32)

    def _apply_action(self, action):
        a = np.clip(action, -1.0, 1.0)
        if abs(a[0]) > 0.5:
            self._lp['with_tie']   = not self._lp['with_tie']
        if abs(a[1]) > 0.5:
            self._lp['with_dummy'] = not self._lp['with_dummy']
        if abs(a[2]) > 0.5:
            self._lp['placement']  = ('row' if self._lp['placement'] == 'column'
                                      else 'column')
        self._lp['sep_mult'] = float(np.clip(
            self._lp['sep_mult'] + a[3] * 0.5, 1.0, 4.0
        ))
        self._lp['met_layer'] = int(np.clip(
            round(self._lp['met_layer'] + a[4]), 1, 3
        ))
        self._lp['width_mult'] = float(np.clip(
            self._lp['width_mult'] + a[5] * 0.25, 0.5, 3.0
        ))

    # ── DRC helpers ───────────────────────────────────────────────────────────

    def _build_and_drc(self):
        import gelochip.gl.drc_corrector as _dc
        _dc._RL_ACTIVE = True
        try:
            layout = self._build_fn(self._circuit_params, self._lp, '_rl_tmp')
            result = layout.drc(silent=True)
        except Exception as exc:
            _dc._RL_ACTIVE = False
            print(f'  [env] error: {exc}')
            result = {'total_errors': 200,
                      'NW': 40, 'CO': 40, 'PL': 40, 'DF': 40, 'other': 40}
        finally:
            _dc._RL_ACTIVE = False
        return result

    def _drc_to_obs(self, drc):
        cats = np.array([
            drc.get('NW', 0), drc.get('CO', 0), drc.get('PL', 0),
            drc.get('DF', 0), drc.get('other', 0),
        ], dtype=np.float32) / self.MAX_ERRORS
        return np.clip(cats, 0.0, 1.0)

    # ── Gym API ───────────────────────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._build_fn, self._circuit_params, self._current_label = \
            random.choice(self.circuit_pool)
        self._lp   = dict(self.DEFAULT_LP)
        self._step = 0
        drc = self._build_and_drc()
        n   = drc.get('total_errors', 0)
        obs = np.concatenate([self._drc_to_obs(drc), self._lp_obs()])
        return obs, {'drc_errors': n, 'layout_params': dict(self._lp),
                     'circuit': self._current_label}

    def step(self, action):
        self._apply_action(action)
        self._step += 1
        drc = self._build_and_drc()
        n   = drc.get('total_errors', 0)

        reward = -n / self.MAX_ERRORS
        if n == 0:
            reward += 20.0

        if n < self._best_errors:
            self._best_errors       = n
            self.best_layout_params = dict(self._lp)

        obs  = np.concatenate([self._drc_to_obs(drc), self._lp_obs()])
        done = (n == 0) or (self._step >= self.max_steps)
        info = {'drc_errors': n, 'layout_params': dict(self._lp),
                'circuit': self._current_label}
        return obs, reward, done, False, info


# Quick sanity check
env = DRCCorrectorEnv(CIRCUIT_POOL, max_steps=15)
obs, info = env.reset()
print('Env OK')
print(f'  Circuit: {info["circuit"]}')
print(f'  DRC errors: {info["drc_errors"]}')
print(f'  obs shape: {obs.shape}')


2026-05-20 14:29:20.015 | INFO     | gdsfactory.pdk:activate:337 - 'gf180' PDK is now active
2026-05-20 14:29:21.185 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/_rl_tmp.gds'
2026-05-20 14:29:21.211 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/notebooks/layoutRL/_RL_TMP.gds'


using provided pdk_root
Defaulting to stale magic_commands.tcl


2026-05-20 14:29:21.452 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmpl8mntn97/_RL_TMP.gds'



Magic 8.3 revision 644 - Compiled on Sun May 17 10:32:25 PM WIB 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmpcvh3neqp/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0
Library name: library
Reading "_RL_TMP".
[INFO]: Loading _RL_T

## Step 4 — Train PPO

PPO (Proximal Policy Optimization) is the standard RL algorithm for continuous control.

Each timestep = one DRC run (~5–8 seconds with Magic).
- **50 timesteps** ≈ 5–7 min → proof of concept
- **200 timesteps** ≈ 20–30 min → decent policy
- **1000+ timesteps** ≈ 2–3 hours → production quality

> Run this cell, go get coffee, come back.


In [5]:
class DRCProgressCallback(BaseCallback):
    """Print DRC status after each episode."""
    def __init__(self):
        super().__init__()
        self.ep = 0

    def _on_step(self):
        for info in self.locals.get('infos', [{}]):
            if 'drc_errors' in info:
                n   = info['drc_errors']
                lp  = info['layout_params']
                ckt = info.get('circuit', '?')
                status = '✅ CLEAN' if n == 0 else f'❌ {n:3d} errors'
                print(f'  ep {self.ep:4d} [{ckt:20s}]: {status} '
                      f'| placement={lp["placement"]:6s} '
                      f'sep={lp["sep_mult"]:.1f} '
                      f'tie={int(lp["with_tie"])} '
                      f'dum={int(lp["with_dummy"])} '
                      f'met={lp["met_layer"]} '
                      f'w={lp["width_mult"]:.1f}')
                self.ep += 1
        return True


# PPO with deeper network for multi-circuit generalisation
model = PPO(
    'MlpPolicy', env,
    policy_kwargs=dict(net_arch=[128, 128, 64]),
    learning_rate=3e-4,
    n_steps=15,
    batch_size=15,
    n_epochs=5,
    gamma=0.99,
    verbose=0,
)

print('Training DRC corrector...')
print('Each step builds a real layout + runs Magic DRC (~5–8s per step)')
print()

# ── Change total_timesteps for longer training ────────────────────────────────
# 50   = quick test (~5 min)
# 200  = decent policy (~25 min)
# 1000 = production (~2 hr)
TOTAL_TIMESTEPS = 200

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=DRCProgressCallback())
print()
print(f'Training done. Best errors seen: {env._best_errors}')
print(f'Best layout params: {env.best_layout_params}')


Training DRC corrector...
Each step builds a real layout + runs Magic DRC (~5–8s per step)



2026-05-20 14:29:28.319 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/_rl_tmp.gds'
2026-05-20 14:29:28.320 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/notebooks/layoutRL/_RL_TMP.gds'


using provided pdk_root
Defaulting to stale magic_commands.tcl


2026-05-20 14:29:28.551 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmput1imqdl/_RL_TMP.gds'



Magic 8.3 revision 644 - Compiled on Sun May 17 10:32:25 PM WIB 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmpp7bcb2op/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0
Library name: library
Reading "_RL_TMP".
[INFO]: Loading _RL_T

...


Magic 8.3 revision 644 - Compiled on Sun May 17 10:32:25 PM WIB 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmpzspa6hwu/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0
Library name: library
Reading "_RL_TMP".
[INFO]: Loading _RL_T

2026-05-20 14:44:45.118 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/_rl_tmp.gds'
2026-05-20 14:44:45.119 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/notebooks/layoutRL/_RL_TMP.gds'


using provided pdk_root
Defaulting to stale magic_commands.tcl

Magic 8.3 revision 644 - Compiled on Sun May 17 10:32:25 PM WIB 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmptmmltsmd/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0

2026-05-20 14:44:45.332 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmp9w07w1b5/_RL_TMP.gds'


using default pdk_root
  ep  209 [tgate               ]: ✅ CLEAN | placement=row    sep=1.7 tie=1 dum=1 met=1 w=1.1

Training done. Best errors seen: 0
Best layout params: {'with_tie': True, 'with_dummy': False, 'placement': 'row', 'sep_mult': 2.1305570658296347, 'met_layer': 1, 'width_mult': 1.5}


## Step 5 — Save model to `src/gelochip/gl/`

Saves a single `drc_corrector_ppo.pt` file containing:
- `policy_state_dict` — trained PPO network weights (PyTorch)
- `best_layout_params` — best layout params seen during training (fallback)

Once saved, `gl.auto_build()` loads it automatically via `torch.load()`.


In [6]:
import torch

SAVE_PATH = os.path.join('..', '..', 'src', 'gelochip', 'gl', 'drc_corrector_ppo')

# Save policy weights + best layout params in a single .pt file
torch.save({
    'policy_state_dict': model.policy.state_dict(),
    'best_layout_params': env.best_layout_params,
}, SAVE_PATH + '.pt')

print(f'Model saved → {SAVE_PATH}.pt')
print(f'Best errors seen: {env._best_errors}')
print(f'Best layout params: {env.best_layout_params}')
print()
print('Done! You can now use:')
print()
print('  chip = gl.auto_build(build_inverter, {"wn":2, "wp":4, "fn":2}, name="inv")')


Model saved → ../../src/gelochip/gl/drc_corrector_ppo.pt
Best errors seen: 0
Best layout params: {'with_tie': True, 'with_dummy': False, 'placement': 'row', 'sep_mult': 2.1305570658296347, 'met_layer': 1, 'width_mult': 1.5}

Done! You can now use:

  chip = gl.auto_build(build_inverter, {"wn":2, "wp":4, "fn":2}, name="inv")
